# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** A page is worth flagging if its CTR is well below
what pages at the same average search position typically get, AND it's getting
enough impressions that fixing it would actually matter. Low CTR on a page
nobody sees isn't worth the team's time; low CTR on a high-traffic page is.

In [1]:
# signal 1: CTR vs. position, the flag-linked signal
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/Michael-AI-Dam/Flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv")

# Guard: avg_position == 0 means "no data", not rank zero — exclude those rows
df_valid = df[df["avg_position"] > 0].copy()

# Signal 1: CTR relative to position-tier peers (behind the CTR-fix logic flag)
df_valid["ctr_by_position_avg"] = df_valid.groupby("position_tier")["ctr"].transform("mean")
df_valid["ctr_below_peers"] = df_valid["ctr"] < df_valid["ctr_by_position_avg"]

bucket_1 = df_valid.groupby("ctr_below_peers").size().reset_index(name="n")
print(bucket_1)
print("\nVerdict: CONFIRMED — clear split between pages above vs below peer CTR at their position.")

   ctr_below_peers      n
0            False   5024
1             True  23771

Verdict: CONFIRMED — clear split between pages above vs below peer CTR at their position.


In [2]:
# Signal 2: impressions volume (behind the quick-win flag)
impressions_threshold = df_valid["impressions_last_30d"].median()
df_valid["high_volume"] = df_valid["impressions_last_30d"] >= impressions_threshold

bucket_2 = df_valid.groupby("high_volume").size().reset_index(name="n")
print(bucket_2)
print("\nVerdict: CONFIRMED — roughly half the pages clear the median volume threshold, a usable split.")

   high_volume      n
0        False  14375
1         True  14420

Verdict: CONFIRMED — roughly half the pages clear the median volume threshold, a usable split.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The rule combines both signals into a transparent score: a page only scores if it
has both low relative CTR AND high volume — multiplying (not adding) means a page
needs both conditions true to rank, not just one.

In [4]:
import os

# Transparent score — readable on purpose, no fitted weights
low_ctr = df_valid["ctr_below_peers"].astype(int)
high_impr = df_valid["high_volume"].astype(int)
gap = (df_valid["ctr_by_position_avg"] - df_valid["ctr"]).clip(lower=0)
No
df_valid["score"] = low_ctr * high_impr * gap * df_valid["impressions_last_30d"]

# Reason codes
def reason_code(row):
    if row["ctr_below_peers"] and row["high_volume"]:
        return "low_ctr_high_volume"
    elif row["ctr_below_peers"]:
        return "low_ctr_only"
    elif row["high_volume"]:
        return "high_volume_only"
    else:
        return "no_flag"

df_valid["reason_code"] = df_valid.apply(reason_code, axis=1)
df_valid["action"] = np.where(df_valid["score"] > 0, "review_title_and_metadata", "no_action")

ranked = df_valid.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
ranked[["content_id", "score", "reason_code", "action", "ctr", "ctr_by_position_avg",
        "impressions_last_30d"]].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(ranked))
ranked[["content_id", "score", "reason_code", "action"]].head(10)

NameError: name 'No' is not defined

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.